# Tutorial 07: grid problem and car braking

In this tutorial, we will look at two implementations of dynamic programming in discrete systems. We will use concepts such as Bellman's optimality principle to implement optimal controllers.

**Pre-requisites**

Basic notions of Dynamic Programming.

**Goals**

Learning to implement Dynamic Programming in discrete problems, building up the notions of trajectory optimization.

This notebook is organized as follows:

    1. Recap: Dynamic Programming
        1.1. The value iteration algorithm
    2. Grid world
        2.1. Problem setup
        2.2. Solving the problem using value iteration
    3. Braking car
        3.1. Problem setup
        3.2. Solving the problem using value iteration

## 1. Recap: Dynamic Programming

One of the most prominent uses of Dynamic Programming is to solve optimizations problems recursively. The main idea behind it can be summarized using Bellman's principle of optimality:

>An optimal policy has the property that whatever the initial state and initial decision are, the remaining decisions must constitute an optimal policy with regard to the state resulting from the first decision.

That is to say that, if we know that a trajectory with a starting and an ending state is optimal, the remaining decisions will be in this trajectory regardless of which intermediate decision we are looking at.
This concept takes us to the following expression:

$$ \forall i \quad
    J^*(\mathbf{x}_i) \Leftarrow \min_{\mathbf{u} \in \mathcal{U}} \left[ \ell(\mathbf{x}_i,\mathbf{u}) +
    J^*\left({f(\mathbf{x}_i,\mathbf{u})}\right) \right].
$$

where:
- $J^*(\mathbf{x}_i)$ is the optimal cost-to-go at state $\mathbf{x}_i$.
- $\ell(\mathbf{x}_i,\mathbf{u})$ is the cost associated to the performing a certain action, $\mathbf{u}$ at a certain state. Note that $\mathbf{u}$ must be in the set of allowed actions, $\mathcal{U}$.
- $J^*\left({f(\mathbf{x}_i,\mathbf{u})}\right)$ is the optimal cost-to-go associated to the state we reach after updating our state using the update function $f(\mathbf{x},\mathbf{u})$.

### 1.1. The value iteration algorithm

So far, for Bellman's principle of optimality to apply, we must know the optimal solution beforehand. However, we can also leverage its definition to solve trajectory optimization problems. First, we initialize the estimate of the optimal cost-to-go to zero for all states and provide a feasible initial guess for our actions. Then, we repeat the following two steps:

- __Policy evaluation__: Update the optimal cost-to-go estimate with the current set of actions.
- __Policy update__: Use the new optimal cost-to-go estimate to adjust the policy.
$$
\pi(\mathbf{x}_i) = \text{argmin}_\mathbf{u}
\left[ \ell(\mathbf{x}_i,\mathbf{u}) + \hat{J}^*\left( f(\mathbf{x}_i,\mathbf{u}) \right) \right].
$$

Here, $\hat{J}^*$ is the estimated optimal cost-to-go. When the optimal cost-to-go for all states has converged, we can simply trace the trajectory using the optimal policy, $\pi^*(\mathbf{x}_i)$.

# 2. Grid world

Now, let's put Dynamic Programming to the test with a classical problem and have it compute the optimal route to the to a point around obstacles.

## 2.1. Problem setup

In this problem, we find ourselves in a discrete two-dimensional world. Each second, we can choose to move up, down, left, right, or stay where we are. We want to get to the goal as fast as possible. Additionally, there are some obstacles we should avoid. First, we will import the necessary modules.

In [ ]:
import numpy as np
from grid import Grid

Time to instantiate the Grid class. We will now define the size of our grid world and the location of our goal.

In [ ]:
gridsize = [15, 10]
goal = [2, 2]

world = Grid(gridsize, goal)
world.drawMap()

If everything went well, we should see a map of the cost $\ell(x)$ associated to stepping on each tile. It should be zero for the goal and 1 for the rest. To make things more interesting, let's add an obstacle. In this class, rectangular obstacles are defined using the _addObstacle()_ method. Its arguments are:

- A two-position vector containing the minimum and maximum values of x that the obstacle covers.
- A two-position vector containing the minimum and maximum values of y that the obstacle covers.
- The penalty associated to stepping on a cell covered by the obstacle.

Let's try adding an obstacle to our map:

In [ ]:
obstacle_x = [1,6]
obstacle_y = [5,8]
obstacle_cost = 5

world.addObstacle(obstacle_x, obstacle_y, obstacle_cost)
world.drawMap()

We also define the update function, $f(x,u)$. _updatePosition_ is a method that takes the following arguments:
- Current position in x.
- Current position in y.
- Action in form of a string. "s" for stay, "u" for up, "d" for down, "r" for right, and "l" for left.

In [ ]:
def updatePosition(self, X, Y, move):
    if move == "s": # Stay
        Xnext = X
        Ynext = Y
    elif move == "u": # Move up
        Xnext = X 
        Ynext = Y - 1
    elif move == "d": # Move down
        Xnext = X 
        Ynext = Y + 1
    elif move == "r": # Move right
        Xnext = X + 1 
        Ynext = Y
    elif move == "l": # Move left
        Xnext = X - 1 
        Ynext = Y
    return (Xnext, Ynext)

Grid.updatePosition = updatePosition

## 2.2. Solving the problem using value iteration

In this section, we will implement the value iteration algortihm to obtain the optimal policy to reach the goal as quick as possible. First, we will deal with policy evaluation. In this step, we go over all the cells in the map one by one and estimate their cost-to-go based on the current policy and the cost-to-go estimate of the cell the action will take us to.

In [ ]:
def evaluatePolicy(self):
    # Calculate cost associated to each cell following the current policy

    ctg_map = np.zeros((self.sizeY, self.sizeX))

    
    for j in range(0, self.sizeY):
        for i in range(0, self.sizeX):

            ## Type here!
            
            Xnext, Ynext = [0.0, 0.0]
            ctg_map[j][i] = 0.0
    
    err = 0.0
            ##
    
    self.cost_to_gomap = ctg_map

    return err

Grid.evaluatePolicy = evaluatePolicy

## Think-Pair-Share (5 min)

Complete the implementation of the policy evaluation step of the value iteration algorithm. 
- __Hint__: Use the _self.updatePosition_ method to get the position you get from applying an action. _self.policy_ is an array that contains the current policy associate to every cell.
- __Hint__: _self.cost_to_go_map_ contains the previous cost-to-go map. Use it to generate the updated cost-to-go map but don't modify it. It will help us know when our solution has converged and we can stop iterating.
- __Hint__: The array _self.costmap_ contains the scalar cost of being in a cell for a second ($\ell(x)$).

Now, we will complete the implementation of the policy update step.

In [ ]:
def updatePolicy(self):
    # Find reachable cell with minimal cost-to-go

    for j in range(0, self.sizeY):
        for i in range(0, self.sizeX):
            c = [np.inf, np.inf, np.inf, np.inf, np.inf] # cost-to-go up, down, right, left, stay
            
            ## Type here!
            if j>0:
                # up
                c[0] = 0.0
            
            if j<(self.sizeY-1):
                # down
                c[1] = 0.0

            if i>0:
                # right
                c[2] = 0.0

            if i<(self.sizeX-1):
                #left
                c[3] = 0.0

            # stay
            c[4] = 0.0

            J_star = 0.0 # Optimal cost-to-go
            ##

            # Find value of J_star in vector of cost-to-go associated to each action
            self.policy[j][i] = self.directions[c.index(J_star)]
            
Grid.updatePolicy = updatePolicy

## Think-Pair-Share (5 min)

Complete the implementation of the policy update step of the value iteration algorithm. 
- __Hint__: Note that you may not exit the map. The conditional statements make sure that only feasible actions can be executed.

Lastly. we will combine the two steps of the algorithm to solve it. When the cost-to-go map stops changing, we will know that we have converged.

In [ ]:
def solveDiscreteProblem(self, anim = False):
    self.policy = np.array([["s"]*self.sizeX]*self.sizeY) # Initially the policy associated to all quadrants is "stay"
    self.cost_to_gomap = np.zeros((self.sizeY, self.sizeX))

    err = np.inf
    i = 0

    while i < self.maxiter and err > self.tolerance:
        ## Type here!
        err = 0.0
        
        ##
        i += 1

    print("iterations : ", i)
    print("error: ", err)

Grid.solveDiscreteProblem = solveDiscreteProblem

## Think-Pair-Share (5 min)

- Complete the implementation of the main loop of the value iteration algorithm.
- __Hint__: You must call both _self.evaluatePolicy_ and _self.updatePolicy_ in the main loop of the algorithm.

Let's solve our problem. Run the following cell and see if the algorithm successfully solves the problem.

In [ ]:
gridworld = Grid(gridsize, goal)
gridworld.addObstacle(obstacle_x, obstacle_y, obstacle_cost)
gridworld.maxiter = 100
gridworld.solveDiscreteProblem()
gridworld.drawCTGMap()

If everything worked well you should see the following map:

<div style="display: flex; justify-content: space-around;">
    <div><img src="gridsolution.png" width="600"></div>
</div>

The color of the cell shows its optimal cost-to-go, and the blue arrows show the optimal policy. To go from any cell to the goal in an optimal manner, just follow the arrows from the starting cell to the goal.

## Think-Pair-Share (10 min)

If you did not get the expected result, think about why. You can see the evolution of the convergence by lowering _gridworld.maxiter_ to stop the algortihm prematurely. Use this to debug if necessary. 
- If your code worked but you didn't get the exact same policy think about why. Is it possible to have multiple optimal solutions?
- Try adding more obstacles, shifting the goal, etc. and run the algorithm again.

# 3. Braking car

Now we will move onto a slightly more complex application case for Dynamic Programming. This problem involves a car whose goal it is to stop at a certain point of a road.

## 3.1. Problem setup

Our car is traveling down a road. Its state, $\mathbf{x}$, comprises its velocity and position, such that:

$$
\mathbf{x} = \begin{bmatrix}
x \\
\dot{x}
\end{bmatrix}
$$

This system has been simplified to a linear system. The update function is as follows:

$$
\mathbf{x}_{i+1} =  \mathbf{A}
\mathbf{x} + \mathbf{B} \mathbf{u} = \begin{bmatrix}
 1 & 1 \\
 0 & 1
\end{bmatrix} 
\mathbf{x} + \begin{bmatrix}
0 \\
1
\end{bmatrix} \mathbf{u}
$$

Our control variable, $\mathbf{u}$ is the acceleration of the car. This time, we want to get as close as possible to the desired end-state as quick as possible, but we also want to minimize the acceleration. With this balance in mind, we propose the following cost function:

$$
J = \int_{t_f}^{0} \ell(\mathbf{x}, \mathbf{u}) dt = \int_{t_f}^{0} (\mathbf{x}^T\mathbf{Q}\mathbf{x} + \mathbf{u}^T\mathbf{R}\mathbf{u}) dt
$$

Before we start solving the problem, we will have to import our main class and set up the problem parameters.

In [ ]:
import numpy as np
from car_braking import CarBrakingDiscrete
from set_theory_utils import precursorSet, findFeasibleSet

qmin = -10 # Min. pos. [m]
qmax = 10  # Max. pos. [m]
qdmin = -3 # Min. vel. [m/s]
qdmax = 3  # Min. vel. [m/s].
umin = -2  # Min. acceleration [m/s^2]  
umax = 2   # Max. acceleration [m/s^2]

A = np.array([[1,1],[0,1]])
B = np.array([0,1])
Q = np.array([[1,0], [0,1]])
R = np.array([1])

To make our lives a bit easier, we have automated finding the feasible set of actions. There are many ways to deal with the system being driven outside of the bounds of our problem, this is just one of them.

In [ ]:
goal_set = np.zeros((int((qdmax-qdmin)+1), int((qmax-qmin)+1)), dtype=np.int64)
goal_set[int(-qdmin)][int(-qmin)] = 1
S, u_feas = findFeasibleSet(goal_set, [qmin, qmax, qdmin, qdmax], A, B, [umin, umax])

## 3.2. Solving the problem using value iteration

Now that we know how the value iteration algorithm works, we will introduce the more complex cost functions and system dynamics described in subsection 3.1.

In [ ]:
def cost_function(self,x,u):
    # cost associated to action and state in one instant
    
    cost = 0.0 # Type here!

    return cost

def update_position(self, x, u):
    # Update position and velocity of the vehicle (discrete coordinates)
    
    xnew = np.array([0.0, 0.0]) # Type here!
    return xnew

CarBrakingDiscrete.cost_function = cost_function
CarBrakingDiscrete.update_position = update_position

## Think-Pair-Share (5 min)

Complete the previous cell, then run the next one. Debug your code if necessary. Like before, a particularly useful tool is the iteration limit, which can allow you to stop the algorithm at any point to check how the cost-to-go and policy are eveolving.

In [ ]:
car = CarBrakingDiscrete()
car.setGoal([0,0])
car.setPosLimit(qmin, qmax)
car.setVelLimit(qdmin, qdmax)
car.setULimit(umin, umax)
car.setCostFunction(Q, R)
car.setAllowedActions(u_feas)
car.maxiter = 100
car.generateMeshes()
car.solveDiscreteproblem()
car.drawMap()
car.drawPolicy()

## Think-Pair-Share (15 min)

- Play around with the values of $\mathbf{Q}$ and $\mathbf{R}$, what changes do you see in the results?
- Modify the values of the minimum and maximum positions, velocities and accelerations and discuss the new results

So far, we have only discussed discrete systems in which the only allowed actions will drive the system to one of a few pre-determined states. If you have time:
- How would you modify the code to be able to use discrete states and inputs without needing to "land" in predefined states?
- What connections can you see to the Hamilton-Jacobi-Bellman equation?

$$
0 = \min_\mathbf{u} \left[
      \ell(\mathbf{x},\mathbf{u}) + \frac{\partial J^*}{\partial \mathbf{x}}f(\mathbf{x},\mathbf{u}) \right], \\
$$